# 3GPP Rel-18 RAG — Quickstart Notebook

**Open Telco RAG**

This notebook walks through the complete Open Telco RAG pipeline for the
**3GPP Release 18** corpus — files from
the [TSpec-LLM](https://huggingface.co/datasets/netop-team/TSpec-LLM) dataset.

By the end of this notebook you will have:
- Downloaded and ingested 3GPP Rel-18 Markdown files into a local ChromaDB
- Run semantic search with clause-level precision
- Asked natural language questions about intent management and autonomous networks
- Evaluated retrieval quality with the evaluation metrics

---

### Prerequisites

```bash
git clone https://github.com/Znbne/telco-retrieve
cd open-telco-rag
pip install -e .
```

Set your API keys in a `.env` file at the project root:
```
# For answer generation (choose one or both)
ANTHROPIC_API_KEY=sk-ant-...        # Claude models direct
OPENROUTER_API_KEY=sk-or-...        # Any model via OpenRouter
```

## 1. Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# TSpec-LLM Rel-18 Markdown files
DATA_DIR   = Path("data/3gpp/TSpec-LLM/3GPP-clean/Rel-18")
DB_PATH    = "chroma_db"
MODEL_KEY  = "minilm"
COLLECTION = f"3gpp_{MODEL_KEY}"

print(f"Data directory : {DATA_DIR}")
print(f"ChromaDB path  : {DB_PATH}")
print(f"Embedding model: {MODEL_KEY}")
print(f"Collection     : {COLLECTION}")

## 2. Download 3GPP Rel-18 Specifications

3GPP specifications are available via the **TSpec-LLM** dataset, which provides
all 3GPP documents from Release 8 to Release 19 converted to Markdown using pandoc.

Download the dataset from HuggingFace:
```bash
pip install huggingface_hub
python -c "from huggingface_hub import snapshot_download; snapshot_download('netop-team/TSpec-LLM', repo_type='dataset', local_dir='data/3gpp')"
```

Or download Release 18 only:
```bash
python scripts/download_3gpp.py --release 18 --output data/3gpp
```

The cell below checks how many files are available.

In [ ]:
# Count available 3GPP Rel-18 Markdown files
if DATA_DIR.exists():
    files = list(DATA_DIR.rglob("*.md"))
    print(f"Total 3GPP Rel-18 files: {len(files)}")
    print("\nSample filenames (spec_number-version.md):")
    for f in sorted(files)[:5]:
        print(f"  {f.name}")
    print("\nFilename convention:")
    print("  28312-i11.md  →  TS 28.312, Release 18 (i=Rel-18), version 11")
else:
    print(f"[INFO] {DATA_DIR} not found.")
    print("Download TSpec-LLM from: https://huggingface.co/datasets/netop-team/TSpec-LLM")

## 3. Ingest into ChromaDB

The ingest pipeline for 3GPP:
1. Loads each Markdown file using `TPPSource`
2. Removes pandoc artefacts (pipe tables, image refs, HTML residuals)
3. Applies **heading-aware chunking** — splits on `#`, `##`, `###` boundaries
4. Extracts metadata: spec number, release, series, version
5. Embeds each chunk with `all-MiniLM-L6-v2` and stores in ChromaDB

**Heading-aware chunking** preserves document structure, enabling clause-level
retrieval — answers can cite specific section numbers.

In [ ]:
from open_telco_rag.sources.tgpp import TPPSource
from open_telco_rag.embedders.sentense_transformers import SentenceTransformerEmbedder
from open_telco_rag.stores.chroma import ChromaStore
from open_telco_rag.core.pipeline import ingest

source   = TPPSource()
embedder = SentenceTransformerEmbedder(model_key=MODEL_KEY)
store    = ChromaStore(path=DB_PATH, collection_name=COLLECTION)

print(f"Starting ingest — {store.count()} chunks already in collection")

if DATA_DIR.exists():
    stats = ingest(
        source     = source,
        store      = store,
        embedder   = embedder,
        data_dir   = str(DATA_DIR),
        chunk_size = 1024,
        overlap    = 200,
        resume     = True,
    )
    print(f"\nIngest complete")
    print(f"  Files processed : {stats.files_processed}")
    print(f"  Chunks stored   : {stats.chunks_stored}")
    print(f"  Skipped         : {stats.skipped_files}")
    print(f"  Errors          : {stats.error_files}")
print(f"  Total in DB     : {store.count():,}")

## 4. Explore the Collection

3GPP metadata includes spec number, series, and release —
enabling series-filtered retrieval across the 40+ spec series.

In [ ]:
import chromadb
import statistics
from collections import Counter

client     = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection(COLLECTION)
sample     = collection.get(limit=2000, include=["documents", "metadatas"])

series_counts = Counter(m.get("series", "?") for m in sample["metadatas"])
lengths       = [len(d) for d in sample["documents"]]

SERIES_DESC = {
    "28": "Network management & orchestration",
    "38": "5G NR radio access",
    "23": "Architecture and procedures",
    "32": "Telecom management",
    "33": "Security",
    "24": "Core network protocols",
    "36": "LTE (E-UTRAN)",
    "29": "Core network interfaces",
}

print(f"Collection: {COLLECTION}")
print(f"Total chunks: {collection.count():,}\n")
print(f"Top series by chunk count (sample of 2,000):")
print(f"{'Series':>8} {'Chunks':>8}  Description")
print("-" * 50)
for series, count in series_counts.most_common(10):
    desc = SERIES_DESC.get(series, "")
    print(f"{series:>8} {count:>8}  {desc}")

print(f"\nChunk length statistics (chars):")
print(f"  Mean   : {statistics.mean(lengths):.0f}")
print(f"  Median : {statistics.median(lengths):.0f}")
print(f"  Min    : {min(lengths)}")
print(f"  Max    : {max(lengths)}")

## 5. Semantic Search

Run a semantic search query. 3GPP Markdown chunks preserve section headings,
so retrieved chunks include the clause context they came from.

In [ ]:
from open_telco_rag.core.pipeline import search

QUERY = "What does TS 28.312 define about intent management?"
TOP_K = 5

results = search(
    QUERY,
    store    = store,
    embedder = embedder,
    top_k    = TOP_K,
)

print(f"Query: '{QUERY}'")
print(f"Top-{TOP_K} results:\n")
print(f"{'Rank':<5} {'Similarity':>10} {'Spec':>10} {'Series':>8} {'Source'}")
print("-" * 75)

for i, r in enumerate(results, 1):
    sim    = round(1 - r.distance, 3)
    spec   = r.metadata.get("spec_number", "")
    series = r.metadata.get("series", "")
    src    = r.metadata.get("source", "")[:30]
    print(f"{i:<5} {sim:>10.3f} {spec:>10} {series:>8} {src}")

print(f"\nTop result preview:")
print(results[0].text[:400])

### 5.1 Cross-series retrieval

3GPP knowledge is distributed across many series. Autonomous network level
definitions span the 28-series (management) and 32-series (architecture).
Run the same query and observe which series are retrieved.

In [ ]:
QUERY2 = "What are the 3GPP autonomous network level definitions?"

results2    = search(QUERY2, store=store, embedder=embedder, top_k=8)
series_hit  = Counter(r.metadata.get("series", "?") for r in results2)
mean_sim    = statistics.mean(1 - r.distance for r in results2)

print(f"Query: '{QUERY2}'")
print(f"Series retrieved: {dict(series_hit.most_common())}")
print(f"Mean similarity : {mean_sim:.3f}")
print(f"\n{'Rank':<5} {'Similarity':>10} {'Spec':>10} {'Series':>8}")
print("-" * 40)
for i, r in enumerate(results2, 1):
    print(f"{i:<5} {1-r.distance:>10.3f} {r.metadata.get('spec_number',''):>10} {r.metadata.get('series',''):>8}")

## 6. Question Answering

Ask a natural language question. The 3GPP Markdown format enables
clause-level citation accuracy — Claude can reference specific clause numbers.

In [ ]:
from open_telco_rag.core.pipeline import generate

QUESTION = "What does TS 28.312 define about intent management?"

retrieved = search(QUESTION, store=store, embedder=embedder, top_k=5)

from open_telco_rag.generators.openrouter import OpenRouterGenerator
generator = OpenRouterGenerator(model="meta-llama/llama-3.3-70b-instruct")

response  = generate(QUESTION, retrieved, source_type="3gpp", generator=generator)

print(f"Question: {QUESTION}")
print(f"{'─'*70}")
print(f"\n{response['answer']}")
print(f"\n{'─'*70}")
print(f"Sources used: {response['sources']}")
print(f"Chunks used : {response['chunks_used']}")

### 6.1 Question bank

Try the full 3GPP evaluation question set across all relevant series.

In [ ]:
QUESTIONS = [
    ("What does TS 28.312 define about intent management?",            "28"),
    ("What are the 3GPP autonomous network level definitions?",         "32"),
    ("How does 3GPP define network slicing management?",               "28"),
    ("What is NWDAF and what does it provide in 5G?",                  "23"),
    ("What are the requirements for energy efficiency in 5G NR?",      "38"),
    ("How does SON work in 3GPP network management?",                  "32"),
    ("What is the 3GPP management data analytics service?",            "28"),
    ("What does 3GPP specify for RAN intelligence and optimisation?",  "38"),
    ("How does 3GPP define the SMO framework for O-RAN?",              "28"),
    ("What are the 3GPP security requirements for network slicing?",    "33"),
]

print(f"{'#':<3} {'Expected series':<17} {'Top series retrieved':<22} {'Mean sim':>9} {'Question'}")
print("-" * 95)

for i, (question, expected_series) in enumerate(QUESTIONS, 1):
    results  = search(question, store=store, embedder=embedder, top_k=5)
    sims     = [1 - r.distance for r in results]
    mean_sim = sum(sims) / len(sims) if sims else 0
    sc       = Counter(r.metadata.get("series", "?") for r in results)
    top_s    = sc.most_common(1)[0][0] if sc else "?"
    hit      = "✓" if top_s == expected_series else "✗"
    q_short  = question[:45] + "..." if len(question) > 45 else question
    print(f"{i:<3} {expected_series:<17} {top_s + ' ' + hit:<22} {mean_sim:>9.3f} {q_short}")

## 7. Evaluation

Run the evaluation metrics across all 3GPP evaluation questions.

**Note**: metric scoring calls the LLM API (configured below).
The full run takes approximately 3–5 minutes for 10 questions.

In [ ]:
from open_telco_rag.evaluation.evaluate import evaluate_corpus
from open_telco_rag.evaluation.questions import load_questions

questions = load_questions("3gpp")
print(f"Loaded {len(questions)} evaluation questions for 3GPP")
print(f"Questions with reference answers: {sum(1 for q in questions if q.reference_answer)}")

In [ ]:
from open_telco_rag.evaluation._claude import make_llm_call

# RAG generation model (answers questions)
rag_llm_call = make_llm_call("openrouter", "meta-llama/llama-3.3-70b-instruct")

# Metrics evaluation model (scores faithfulness, relevancy, etc.)
llm_call = make_llm_call("openrouter", "anthropic/claude-sonnet-4.5")

result = evaluate_corpus(
    corpus       = "3gpp",
    store        = store,
    embedder     = embedder,
    questions    = questions,
    model        = MODEL_KEY,
    top_k        = 5,
    delay        = 1.0,
    llm_call     = llm_call,
    rag_llm_call = rag_llm_call,
)


In [ ]:
print(f"\nPer-question results — 3GPP ({MODEL_KEY}, top-k=5)")
print(f"{'#':<3} {'Faith':>6} {'Relev':>6} {'Sim':>6} {'Recall':>7} {'Correct':>8} {'Question'}")
print("-" * 95)

for i, qr in enumerate(result.question_results, 1):
    q_short = qr.question[:45] + "..." if len(qr.question) > 45 else qr.question
    print(
        f"{i:<3} "
        f"{qr.faithfulness:>6.2f} "
        f"{qr.answer_relevancy:>6.2f} "
        f"{qr.mean_similarity:>6.3f} "
        f"{max(0, qr.context_recall):>7.2f} "
        f"{max(0, qr.answer_correctness):>8.2f} "
        f"{q_short}"
    )

print("-" * 95)
print(
    f"{'AVG':<3} "
    f"{result.mean_faithfulness:>6.2f} "
    f"{result.mean_answer_relevancy:>6.2f} "
    f"{result.mean_similarity:>6.3f} "
    f"{max(0, result.mean_context_recall):>7.2f} "
    f"{max(0, result.mean_answer_correctness):>8.2f}"
)


## 8. What the metrics tell us

| Metric | Type | Interpretation |
|--------|------|----------------|
| **Faithfulness** | Generation | Are the claims in the answer grounded in the retrieved chunks? |
| **Answer relevancy** | Generation | Does the answer address what was asked? |
| **Mean similarity** | Retrieval | How confident is the retrieval? Higher = better chunk matches |
| **Context recall** | Retrieval+GT | Were the chunks needed to answer the question retrieved? |
| **Answer correctness** | Generation+GT | Is the answer factually correct vs the reference answer? |
